In [ ]:
import os
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def train_logistic_regression_partial_fit(image_paths, mask_paths, batch_size=4, num_workers=2, val_split=0.2):
    train_imgs, val_imgs, train_masks, val_masks = train_test_split(
        image_paths, mask_paths, test_size=val_split, random_state=42
    )

    train_ds = CloudDataset(train_imgs, train_masks)
    val_ds   = CloudDataset(val_imgs, val_masks)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    model = SGDClassifier(
        loss="log_loss",
        max_iter=1,
        learning_rate='optimal',
        verbose=0,
        n_jobs=-1,
    )

    classes = np.array([0, 1])

    print("Training SGDClassifier with partial_fit...")
    for images, masks in tqdm(train_loader, desc="Training", unit="batch"):
        N, C, H, W = images.shape
        images = images.permute(0, 2, 3, 1).reshape(-1, C)
        masks = masks.view(-1)

        X_batch = images.numpy()
        y_batch = masks.numpy()

        # partial_fit: pass classes only for the first batch
        model.partial_fit(X_batch, y_batch, classes=classes)

    print("Evaluating...")

    val_correct = 0
    val_total = 0

    for images, masks in tqdm(val_loader, desc="Validating", unit="batch"):
        N, C, H, W = images.shape
        images = images.permute(0, 2, 3, 1).reshape(-1, C)
        masks = masks.view(-1)

        X_val_batch = images.numpy()
        y_val_batch = masks.numpy()

        preds = model.predict(X_val_batch)

        batch_correct = np.sum(preds == y_val_batch)
        val_correct += batch_correct
        val_total += len(y_val_batch)

        del images, masks, X_val_batch, y_val_batch, preds
        torch.cuda.empty_cache()

    val_acc = val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.4f}")

    return model

In [ ]:
model = train_logistic_regression_partial_fit(
    image_paths, 
    mask_paths, 
    batch_size=4, 
    num_workers=2, 
    val_split=0.2
)

In [ ]:
import joblib

# Save the trained model
def save_model(model, filename="sgd_model.pkl"):
    joblib.dump(model, filename)
    print(f"Model saved to {filename}")

In [ ]:
save_model(model, "trained_sgd_model.pkl")

In [ ]:
def load_model(filename="sgd_model.pkl"):
    model = joblib.load(filename)
    print(f"Model loaded from {filename}")
    return model

In [ ]:
def predict_mask(model, image_path, device='cpu'):
    with rasterio.open(image_path) as src:
        img = src.read().astype(np.float32)
    img /= img.max()

    img_tensor = torch.from_numpy(img).unsqueeze(0)
    img_flat = img_tensor.permute(0, 2, 3, 1).reshape(-1, img_tensor.shape[1])

    pred = model.predict(img_flat.numpy())
    pred_mask = pred.reshape(img_tensor.shape[2], img_tensor.shape[3])

    return pred_mask

In [ ]:
import random

image_idx = random.randint(0, len(image_paths))
print(f"Image with index: {image_idx}")

model = load_model("trained_sgd_model.pkl")

pred = predict_mask(model, image_paths[image_idx], device='cpu')

with rasterio.open(mask_paths[image_idx]) as src:
    true = src.read(1).astype(np.uint8)

dice = evaluate_and_plot(pred, true)
print(f"Dice = {dice:.4f}")